# BalancedFace source inventory and leakage-safe development index

목적: BUPT-BalancedFace(Equalizedface)를 FR 재학습이나 최종 test가 아니라 PCA/PQ development-fit 및 threshold calibration 후보로 준비한다.

JPG archive는 정상 파일로 교체되어 EOF 검사를 통과했다. 다만 Asian/Indian에 가변 해상도 이미지가 포함되므로 공통 정렬·그룹별 coverage 검증 전에는 비활성으로 둔다. 현재 노트북은 112×112 MXNet RecordIO와 `.lst` metadata 경로를 사용하며, RecordIO decoder가 구현되기 전에는 실제 image manifest 또는 임베딩 입력이 완성된 것으로 간주하지 않는다.

In [ ]:
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "research").is_dir() and (candidate / "configs").is_dir():
            return candidate
    raise FileNotFoundError("C:/ronbun project root could not be located")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from research.datasets import (
    build_balancedface_index_bundle,
    inspect_balancedface_sources,
    select_balancedface_index_scope,
    write_balancedface_index_bundle,
)

MODE = "dev"                    # "dev" 또는 "real"; real은 fraction=1.0
DATA_FRACTION = 0.10             # split×group identity 단위 dev 비율
SEED = 42                        # identity split 및 dev subset seed
DEVELOPMENT_FRACTION = 0.80      # 전체 identity의 development/calibration 비율
EXECUTE_STAGE = False            # list·RecordIO·RFW overlap 전체 검증 실행 여부
WRITE_OUTPUTS = False            # 검증된 source index를 기록할지 여부
VERIFY_SOURCE_SHA256 = False     # 빠른 inventory에서 대용량 hash 재계산 여부
VERIFY_RECORDIO_ARCHIVE = True   # tar EOF·property·idx·rec 구조 검증; 끄면 real 금지

BALANCED_ROOT = PROJECT_ROOT / "data" / "raw" / "RFW-balancedface"
LIST_PATH = BALANCED_ROOT / "rec_for_mxnet" / "train_balancedface.lst"
RECORDIO_ARCHIVE = BALANCED_ROOT / "rec_for_mxnet" / "Equalizedface.tar.gz"
RFW_IDENTITIES_PATH = PROJECT_ROOT / "data" / "interim" / "rfw" / "source_identities.txt"
RFW_SUCCESS_PATH = PROJECT_ROOT / "data" / "interim" / "rfw" / "_SUCCESS"
OUTPUT_DIR = PROJECT_ROOT / "data" / "interim" / "balancedface"

if WRITE_OUTPUTS and not EXECUTE_STAGE:
    raise ValueError("WRITE_OUTPUTS=True requires EXECUTE_STAGE=True")
if MODE == "real" and not VERIFY_RECORDIO_ARCHIVE:
    raise ValueError("real mode requires VERIFY_RECORDIO_ARCHIVE=True")
{
    "project_root": str(PROJECT_ROOT),
    "mode": MODE,
    "data_fraction": DATA_FRACTION,
    "execute_stage": EXECUTE_STAGE,
    "write_outputs": WRITE_OUTPUTS,
}

## 1. 물리적 소스 식별

JPG와 RecordIO는 같은 논리적 training dataset의 대체 표현이지만 행 집합이 완전히 같지는 않다. 이미지 수를 합산하거나 row index로 결합하지 않는다. JPG는 정상 archive이지만 alignment/coverage 검증이 아직 없어 이 노트북에서는 RecordIO metadata 경로를 선택한다.

In [ ]:
source_inventory = inspect_balancedface_sources(
    BALANCED_ROOT,
    project_root=PROJECT_ROOT,
    verify_sha256=VERIFY_SOURCE_SHA256,
)
{
    "summary": source_inventory.summary,
    "artifacts": [artifact.__dict__ for artifact in source_inventory.artifacts],
    "jpg_status": "valid_but_alignment_and_group_coverage_audit_required",
}

## 2. RFW overlap 제거 후 development/calibration index

먼저 RFW 노트북이 기록한 전체 `source_identities.txt`와 `_SUCCESS`를 요구한다. prefix 이전 Freebase ID로 교집합을 구하고, 겹치는 identity의 모든 BalancedFace 행을 제거한 다음 그룹별 identity 단위로 development/calibration을 나눈다.

In [ ]:
balanced_bundle = None
scoped_bundle = None
if EXECUTE_STAGE:
    if not RFW_SUCCESS_PATH.is_file() or not RFW_IDENTITIES_PATH.is_file():
        raise FileNotFoundError(
            "Run notebooks/rfw/data_preparation.ipynb with WRITE_OUTPUTS=True first"
        )
    rfw_source_identity_ids = {
        line.strip()
        for line in RFW_IDENTITIES_PATH.read_text(encoding="utf-8").splitlines()
        if line.strip()
    }
    balanced_bundle = build_balancedface_index_bundle(
        LIST_PATH,
        RECORDIO_ARCHIVE,
        PROJECT_ROOT,
        rfw_source_identity_ids=rfw_source_identity_ids,
        seed=SEED,
        development_fraction=DEVELOPMENT_FRACTION,
        strict_official=True,
        verify_recordio_archive=VERIFY_RECORDIO_ARCHIVE,
    )
    scoped_bundle = select_balancedface_index_scope(
        balanced_bundle,
        mode=MODE,
        data_fraction=DATA_FRACTION,
        seed=SEED,
    )
    preparation_summary = scoped_bundle.summary
else:
    preparation_summary = {
        "status": "not_executed",
        "next_action": "Complete RFW outputs, set EXECUTE_STAGE=True, restart/run all",
    }
preparation_summary

## 3. 검증 artifact 기록

기록 대상은 dev subset이 아니라 전체 overlap 제거·development/calibration split bundle이다. `source_row_index`는 RecordIO key가 아니며, 향후 decoder가 `train.idx`/`train.rec` 연계를 별도로 검증해야 한다.

In [ ]:
written_paths = None
if WRITE_OUTPUTS:
    if balanced_bundle is None:
        raise RuntimeError("BalancedFace bundle must be validated before writing")
    if not balanced_bundle.summary["recordio_archive_integrity_verified"]:
        raise RuntimeError("RecordIO archive integrity must be verified before writing")
    written_paths = write_balancedface_index_bundle(
        balanced_bundle,
        OUTPUT_DIR,
        overwrite=False,
    )
    written_paths = {name: str(path) for name, path in written_paths.items()}
written_paths or "WRITE_OUTPUTS=False: no files written"

## 다음 단계와 현재 중단점

- 현재 출력은 RecordIO source index이며 실제 이미지 파일 manifest가 아니다.
- PyTorch 입력으로 사용하려면 검증된 RecordIO decoder/materializer를 별도 구현하고 112×112 crop hash와 그룹별 decode coverage를 기록해야 한다.
- 그 다음에만 development에서 PCA/PQ를 fit하고 calibration에서 threshold를 보정한다.
- BalancedFace를 최종 test 또는 ArcFace/AdaFace/MagFace 재학습 데이터로 사용하지 않는다.